# Lab 06 — Spark Optimization dan Best Practices
**Evidence-driven optimization: plan, shuffle, join, partition, cache, skew, Spark UI**

Notebook ini dirancang untuk environment pelatihan: Spark 3.5.7 standalone, Jupyter, 2 worker, masing-masing 2 core / 2 GB.  
**Prinsip lab:** prediksi → jalankan → amati → jelaskan. Jangan hanya mengejar output akhir.

## Capaian praktik
Optimasi dilakukan sebagai eksperimen terkontrol: **baseline → locate → hypothesis → change one thing → verify**. Fokusnya bukan mencari magic configuration, tetapi menunjukkan evidence.

In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T
import time, json, os, shutil

spark = (
    SparkSession.builder
    .appName("spark-training-lab")
    .master("spark://spark-master:7077")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Spark version :", spark.version)
print("Master        :", spark.sparkContext.master)
print("App name      :", spark.sparkContext.appName)
print("Default parallelism:", spark.sparkContext.defaultParallelism)

## Workload sintetis
Resource lab sengaja kecil. Ubah `N` ke 5 juta jika laptop kelas kuat; default 2 juta agar eksperimen tetap responsif.

In [ ]:
N = 2_000_000
base = (
    spark.range(N, numPartitions=8)
    .withColumn("key", (F.col("id") % 200_000).cast("long"))
    .withColumn("group", (F.col("id") % 100).cast("int"))
    .withColumn("amount", ((F.col("id") * 7919) % 1_000_000).cast("long"))
    .withColumn("payload", F.concat(F.lit("row-"), F.col("id"), F.lit("-xxxxxxxxxxxxxxxxxxxxxxxx")))
)
print("rows:", base.count())
print("partitions:", base.rdd.getNumPartitions())

In [ ]:
def timed(label, fn):
    t0 = time.perf_counter()
    out = fn()
    sec = time.perf_counter() - t0
    print(f"{label}: {sec:.3f}s")
    return sec, out

## Lab 1 — Baseline + evidence sheet (20 menit)
Jalankan workload, lalu isi evidence dari Spark UI: duration, stage, shuffle read/write, max task duration, spill, partitions.

**Rujukan slide:** Deck 06, slides 52–53.

In [ ]:
baseline = base.groupBy("group").agg(F.sum("amount").alias("total"))
baseline.explain("formatted")
baseline_time, _ = timed("baseline", lambda: baseline.count())

| Metric | Baseline | After | Interpretation |
|---|---:|---:|---|
| Job duration | | | |
| Shuffle read | | | |
| Shuffle write | | | |
| Max task duration | | | |
| Spill | | | |
| Partitions | | | |

**Aturan:** jangan melakukan tuning sebelum minimal satu bottleneck punya evidence.

## Lab 2 — Kurangi data sebelum shuffle (20 menit)
Bandingkan wide/full data vs filtered+selected. Pastikan Anda juga memikirkan apakah business semantics masih sama.

**Rujukan slide:** Deck 06, slide 54.

In [ ]:
a = base.groupBy("group").agg(F.sum("amount").alias("total"))
b = (
    base
    .filter(F.col("amount") > 900_000)
    .select("group", "amount")
    .groupBy("group")
    .agg(F.sum("amount").alias("total"))
)

print("--- A ---")
a.explain("formatted")
print("--- B ---")
b.explain("formatted")

ta, _ = timed("A full", lambda: a.count())
tb, _ = timed("B filtered/projected", lambda: b.count())

**Observasi UI:** apakah shuffle bytes turun? jumlah stage berubah? task duration berubah? Jangan menyebut B “lebih baik” jika filter mengubah pertanyaan bisnis yang sebenarnya ingin dijawab.

## Lab 3 — Join strategy: sort-merge vs broadcast (25 menit)
Agar perbandingan terlihat, nonaktifkan auto broadcast sementara untuk versi A, lalu paksa broadcast untuk versi B.

**Rujukan slide:** Deck 06, slide 55.

In [ ]:
dim = spark.range(50_000).select(
    F.col("id").alias("key"),
    (F.col("id") % 20).alias("segment")
)

old_threshold = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
j1 = base.select("key", "group", "amount").join(dim, "key")
j2 = base.select("key", "group", "amount").join(F.broadcast(dim), "key")

print("--- non-broadcast ---")
j1.explain("formatted")
print("--- forced broadcast ---")
j2.explain("formatted")

t1, _ = timed("join non-broadcast", lambda: j1.count())
t2, _ = timed("join broadcast", lambda: j2.count())

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", old_threshold)

Cari `SortMergeJoin`/`BroadcastHashJoin`, `Exchange`, shuffle read/write, dan memory behavior. Diskusikan kapan broadcast berisiko karena serialized size, growth, atau executor memory.

## Lab 4 — Eksperimen `spark.sql.shuffle.partitions` (25 menit)
Gunakan nilai ekstrem untuk melihat terlalu sedikit vs terlalu banyak task.

**Rujukan slide:** Deck 06, slide 56.

In [ ]:
old_shuffle = spark.conf.get("spark.sql.shuffle.partitions")
partition_results = []
for p in [4, 20, 200]:
    spark.conf.set("spark.sql.shuffle.partitions", p)
    out = base.groupBy("group").agg(F.sum("amount").alias("total"))
    t0 = time.perf_counter()
    out.count()
    sec = time.perf_counter() - t0
    partition_results.append((p, round(sec, 3)))

spark.conf.set("spark.sql.shuffle.partitions", old_shuffle)
spark.createDataFrame(partition_results, ["shuffle_partitions", "seconds"]).show()

Di Spark UI bandingkan number of tasks, median/max duration, scheduler overhead, dan spill. Nilai terbaik bergantung workload; jangan ubah hasil kelas menjadi “angka standar perusahaan”.

## Lab 5 — Cache hanya ketika ada reuse (20 menit)

**Rujukan slide:** Deck 06, slide 57.

In [ ]:
filtered = base.filter(F.col("amount") > 500_000).select("key", "group", "amount")
filtered.cache()

materialize_time, _ = timed("cache materialize", lambda: filtered.count())
reuse1_time, _ = timed("reuse count", lambda: filtered.groupBy("group").count().collect())
reuse2_time, _ = timed("reuse avg", lambda: filtered.groupBy("group").agg(F.avg("amount")).collect())

print("Storage level:", filtered.storageLevel)
filtered.unpersist()

Buka **Storage tab**. Apakah semua partition cached? Apa manfaat pada action kedua/ketiga? Apa yang terjadi setelah `unpersist()`? Diskusikan kondisi ketika cache justru menambah pressure.

## Lab 6 — Buat skew dan temukan straggler (25 menit)

**Rujukan slide:** Deck 06, slide 58.

In [ ]:
skew = (
    spark.range(5_000_000, numPartitions=8)
    .withColumn(
        "key",
        F.when(F.col("id") < 3_500_000, F.lit(0))
         .otherwise((F.col("id") % 100_000).cast("long"))
    )
)

skew_result = skew.groupBy("key").count()
skew_result.explain("formatted")
skew_time, _ = timed("skew groupBy", lambda: skew_result.count())

**Tugas di Spark UI:**
1. Cari slow stage.
2. Sort task by duration / shuffle read.
3. Bandingkan median vs max.
4. Identifikasi hot key.
5. Usulkan mitigasi: AQE, isolate/filter abnormal key, pre-aggregate, broadcast bila relevan, atau salting.

Jangan langsung melakukan salting; pilih teknik sesuai penyebab.

## Final challenge — Tulis rekomendasi optimasi berbasis evidence
Gunakan format:
- **Symptom:** apa yang lambat?
- **Evidence:** metric/plan/UI apa yang membuktikan?
- **Hypothesis:** bottleneck utama?
- **Change:** satu perubahan yang dilakukan.
- **Verification:** metric sebelum vs sesudah.
- **Trade-off / risk:** apa konsekuensinya?

Satu rekomendasi yang terbukti lebih bernilai daripada sepuluh config tanpa evidence.